# Mini-Eval — a 20-Prompt Pass/Fail Harness

A tiny, code-based evaluation loop: load test cases from a JSON file, run each prompt against the model, grade the output **pass/fail**, and report the counts. This is the lightweight replacement for the deprecated hosted Evals platform (shutting down Nov 30 2026) — keep your evals **in code**.

Cases live in **`notebooks/eval-cases.json`** (a 3-case sample ships with this notebook; grow it toward 20 for real coverage).

> **TODO:** the sample `eval-cases.json` has **3** cases. Add ~17 more to reach the 20-prompt target before relying on the pass rate. Execution is deferred — needs a valid API key.

## Setup

In [ ]:
import os, getpass, json, re
def _set_env(var):
    if not os.environ.get(var):
        os.environ[var] = getpass.getpass(f"{var}: ")
_set_env("OPENAI_API_KEY")

from openai import OpenAI
client = OpenAI()

MODEL = "gpt-5.5"   # swap to gpt-5.4-mini for cheaper eval runs

## 1. Load the test cases

In [ ]:
from pathlib import Path

CASES_PATH = Path("eval-cases.json")
data = json.loads(CASES_PATH.read_text())
cases = data["cases"]
print(f"Loaded {len(cases)} cases from {CASES_PATH}")
for c in cases:
    print(f"  - {c['id']} [{c['grader']}]")

## 2. Graders

Three simple code-based graders — `contains`, `equals` (normalized), and `regex`. Code graders are deterministic and free; add a model-based grader later if you need fuzzy judgments.

In [ ]:
def normalize(s: str) -> str:
    return " ".join(s.strip().lower().split())

def grade(grader: str, expected: str, output: str) -> bool:
    if grader == "contains":
        return expected.lower() in output.lower()
    if grader == "equals":
        return normalize(expected) == normalize(output)
    if grader == "regex":
        return re.search(expected, output) is not None
    raise ValueError(f"Unknown grader: {grader}")

## 3. Run the harness

In [ ]:
def run_case(case):
    resp = client.responses.create(
        model=MODEL,
        input=case["prompt"],
        reasoning={"effort": "low"},   # pinned for predictable cost
    )
    output = resp.output_text
    passed = grade(case["grader"], case["expected"], output)
    return passed, output

results = []
for case in cases:
    try:
        passed, output = run_case(case)
    except Exception as e:
        passed, output = False, f"ERROR: {e}"
    results.append({"id": case["id"], "passed": passed, "output": output})
    mark = "PASS" if passed else "FAIL"
    print(f"[{mark}] {case['id']}: {output[:80]!r}")

## 4. Report

In [ ]:
n = len(results)
passed = sum(1 for r in results if r["passed"])
failed = n - passed
rate = (passed / n * 100) if n else 0.0

print("=" * 40)
print(f"Total:  {n}")
print(f"Passed: {passed}")
print(f"Failed: {failed}")
print(f"Pass rate: {rate:.1f}%")
print("=" * 40)

if failed:
    print("\nFailures:")
    for r in results:
        if not r["passed"]:
            print(f"  - {r['id']}: {r['output'][:120]!r}")